In [2]:
import torch
import numpy as np 
import datasets
from model.CausalLM import NoraCausalLM
from config.CustomCLMConfig import NoraConfig
from config.ModelSettings import CMSConfig
from transformers import AutoModelForCausalLM
import torch.nn as nn
from transformers import AutoConfig, AutoModelForCausalLM
# from config.ModelSettings import NoraConfig
from model.CausalLM import NoraCausalLM

from torch.utils.data import DataLoader

/root/KartikGoyal/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [4]:
cms_config = CMSConfig(3072, 3, [1,2,3], [3,4,2], nn.GELU)

config = NoraConfig(model_name="llama-3.2-3b", cms_cfg=cms_config)


In [5]:

AutoConfig.register("NORA", NoraConfig)
AutoModelForCausalLM.register(NoraConfig, NoraCausalLM)


In [17]:

model = AutoModelForCausalLM.from_config(config)
checkpoint = torch.load(
    "/root/KartikGoyal/checkpoints/checkpoint_epoch_7.pt",
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 8963.24it/s]


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
model.to(device)

NoraCausalLM(
  (nora): Nora(
    (decoder): LlamaModel(
      (embed_tokens): Embedding(128256, 3072)
      (layers): ModuleList(
        (0-27): 28 x LlamaDecoderLayer(
          (self_attn): LlamaAttention(
            (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
            (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
            (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
            (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          )
          (mlp): LlamaMLP(
            (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
            (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
            (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
            (act_fn): SiLUActivation()
          )
          (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
          (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        )
    

In [8]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B")
tokenizer.pad_token = tokenizer.eos_token


In [8]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

169924608

In [9]:
sum(p.numel() for p in model.nora.cms.parameters() if p.requires_grad)

169924608

In [9]:

dataset = datasets.load_dataset("wikitext", "wikitext-2-raw-v1")
print("Dataset loaded")


Dataset loaded


In [ ]:

def tokenize(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=2048
    )

    return {"input_ids": tokens["input_ids"], "attention_mask":tokens["attention_mask"]}


dataset = dataset.map(tokenize, batched=True)

dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
print("Dataset tokenized and formatted")


Dataset loaded


Map:   0%|          | 0/4358 [00:00<?, ? examples/s]


NameError: name 'tokenizer' is not defined

In [10]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def generate(
    model,
    input_ids,
    attention_mask=None,
    max_new_tokens=50,
    temperature=1.0,
    top_k=None,
    device="cuda"
):
    model.eval()
    model = model.to(device)

    input_ids = input_ids.to(device)

    if attention_mask is None:
        attention_mask = torch.ones_like(input_ids)

    attention_mask = attention_mask.to(device)

    for _ in range(max_new_tokens):

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False   # keep False for now (your Nora may not support caching fully yet)
        )

        # ---- Correct output handling ----
        if hasattr(outputs, "logits"):
            logits = outputs.logits
        elif isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            raise ValueError("Unknown model output format")

        # ---- Next token ----
        next_token_logits = logits[:, -1, :] / temperature

        # ---- Top-k filtering ----
        if top_k is not None:
            values, _ = torch.topk(next_token_logits, top_k)
            min_val = values[:, -1].unsqueeze(-1)
            next_token_logits = torch.where(
                next_token_logits < min_val,
                torch.full_like(next_token_logits, float('-inf')),
                next_token_logits
            )

        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        # ---- Append token ----
        input_ids = torch.cat([input_ids, next_token], dim=1)

        # ---- Update attention mask ----
        next_mask = torch.ones((attention_mask.size(0), 1), device=device)
        attention_mask = torch.cat([attention_mask, next_mask], dim=1)

    return input_ids

In [11]:
def get_non_empty_sample(dataset_split):
    for item in dataset_split:
        text = item["text"].strip()
        if len(text) > 50:   # threshold avoids junk
            return text
    return None

sample_text = get_non_empty_sample(dataset["test"])

print("RAW SAMPLE:\n", sample_text[:200])

RAW SAMPLE:
 Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 2000 . This was followed by a starring role in the play Herons 


In [12]:
prompt = sample_text[:100]  # first 100 chars

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

input_ids = inputs["input_ids"]
attention_mask = inputs['attention_mask']

In [ ]:
generated_ids = generate(
    model,
    input_ids,
    attention_mask,
    max_new_tokens=100,
    temperature=0.001,
    top_k=50,
    device=device
)

output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("\n=== PROMPT ===")
print(prompt)

print("\n=== GENERATED ===")
print(output_text)

print("\n=== EXPECTED ===")
print(sample_text)

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [23]:

train_loader = DataLoader(
    dataset["train"],
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("DataLoader created")

DataLoader created


In [19]:

def param_gen(param_dict):
    for block_name, (frequency, block_params) in param_dict.items():
        
         
        flat = []
        for p in block_params:
            if isinstance(p, (list, tuple)):
                flat.extend(p)
            else:
                flat.append(p)

        yield {
            "params": flat,
            "block": block_name,
            "frequency": frequency
        }


In [20]:
from optimizers.AdamW import CMSAdamW

params = model.nora.cms.get_param_groups()
# optimizer = CMSAdamW(param_gen(params), 1e-4)
optimizer = CMSAdamW(param_gen(params), lr = 1e-3)

In [24]:
from tqdm import tqdm

num_epochs = 10

model.train()
for epoch in range(num_epochs):
    
    total_loss = 0
    num_batches = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)
    
    for batch in progress_bar:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        model_input = input_ids[:,:-1]
        attention_mask = attention_mask[:,:-1]
        labels = input_ids.clone()

        output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        logits = output.logits
        loss = output.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        num_batches += 1
        avg_loss = total_loss / num_batches

        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg_loss': f'{avg_loss:.4f}',
        })

    print(f"\nEpoch {epoch+1}/{num_epochs} complete — Avg Loss: {avg_loss:.4f}\n")


Epoch 1/10:   0%|          | 1/4590 [00:01<1:40:03,  1.31s/it, loss=21.2500, avg_loss=21.2500]

__
__
__
__
__
__
__
__
__
__
__
__
__
__
__
__
__
__


Epoch 1/10:   0%|          | 2/4590 [00:01<1:08:54,  1.11it/s, loss=0.7227, avg_loss=10.9863] 

__
__
__
__
__
__
__
__
__
__
__
__


Epoch 1/10:   0%|          | 3/4590 [00:02<59:10,  1.29it/s, loss=19.3750, avg_loss=13.7826]  

...

__
__
__
__
__
__
__
__
__
__
__
__


Epoch 1/10:   1%|          | 33/4590 [00:21<48:06,  1.58it/s, loss=0.0630, avg_loss=1.3640]

__
__
__
__
__
__
__
__
__
__
__
__


Epoch 1/10:   1%|          | 33/4590 [00:21<50:33,  1.50it/s, loss=0.0630, avg_loss=1.3640]


KeyboardInterrupt: 